# 16. Sub-queries

*Module: 5 — Multi-table · Companion summary: `lectures/16-sub-queries.md` · SQL engine: SQLite (in-memory)*

## What you will study

- nest a SELECT inside the WHERE clause to avoid manual lookups
- use a sub-query as a column expression
- use a sub-query as a derived table in FROM

---
## Setup — build the sample database this notebook runs on
*Run the cell below first: it creates an in-memory SQLite database and the `q()` helper that prints every query's result set.*

In [ ]:
import sqlite3
import textwrap

conn = sqlite3.connect(':memory:')  # throw-away in-memory database
conn.execute("PRAGMA foreign_keys = ON")

def q(sql):
    """Run a query, print formatted SQL, and display the result set as a table."""
    clean_sql = textwrap.dedent(sql).strip()

    # 1. Print formatted SQL query
    print("\n" + "=" * 60)
    print("QUERY:")
    print(textwrap.indent(clean_sql, "  "))
    print("-" * 60)

    cur = conn.execute(sql)
    rows = cur.fetchall()

    if cur.description:  # SELECT / statement returning rows
        headers = [d[0] for d in cur.description]
        str_rows = [[("" if v is None else str(v)) for v in r] for r in rows]

        # Calculate dynamic column widths for clean alignment
        widths = [
            max(len(h), *(len(r[i]) for r in str_rows), 1)
            for i, h in enumerate(headers)
        ] if str_rows else [len(h) for h in headers]

        header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
        sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))

        print(header_line)
        print(sep_line)
        for r in str_rows:
            print(" | ".join(val.ljust(widths[i]) for i, val in enumerate(r)))
        if not str_rows:
            print("(0 rows returned)")
    else:  # DDL / DML (CREATE, INSERT, UPDATE, DELETE)
        affected = cur.rowcount
        print(f"Done (rows affected: {affected})" if affected != -1 else "Done.")

    print("=" * 60)

print('Connection ready:', conn)

## 🧪 Sample data — EMPLOYEE (id, first, last, salary)

In [ ]:
q("CREATE TABLE employee (emp_id TEXT PRIMARY KEY, f_name TEXT, l_name TEXT, salary REAL)")
q("""
INSERT INTO employee VALUES
    ('E1','Raul','Chong',95000),
    ('E2','Rav','Ahuja',75000),
    ('E3','Lakshmi','Katta',85000),
    ('E4','Wei','Chen',60000),
    ('E5','Ana','Silva',78000),
    ('E6','Jack','Smith',55000)
""")
q("SELECT * FROM employee")

## 🧪 The problem: “who earns more than the average salary?”

The two-step manual version (run one query, read 74666.67, paste it):

In [ ]:
q("SELECT ROUND(AVG(salary), 2) AS avg_salary FROM employee")

In [ ]:
q("SELECT f_name, l_name, salary FROM employee WHERE salary > 74666.67")

## 🧪 Sub-query in WHERE: one self-contained statement

The inner query runs first; its single value replaces the pasted number:

In [ ]:
q("""
SELECT f_name, l_name, salary
FROM employee
WHERE salary > (SELECT AVG(salary) FROM employee)
""")

## 🧪 What breaks: an aggregate directly in WHERE (no nesting)

In [ ]:
# SELECT f_name, salary FROM employee WHERE salary > AVG(salary)   # ❌

In [ ]:
import sqlite3
try:
    conn.execute("SELECT f_name, salary FROM employee WHERE salary > AVG(salary)")
except sqlite3.OperationalError as e:
    print(repr(e))   # aggregates can't appear in WHERE — they need the inner query

## 🧪 Sub-query as a column expression — show the average beside each row

In [ ]:
q("""
SELECT f_name, l_name, salary,
       (SELECT ROUND(AVG(salary), 2) FROM employee) AS avg_company_salary
FROM employee
""")

## 🧪 Sub-query as a derived table — compute in FROM, filter outside

In [ ]:
q("""
SELECT avg_sal
FROM (SELECT ROUND(AVG(salary), 2) AS avg_sal FROM employee) AS avg_table
WHERE avg_sal > 70000
""")

## 💡 Tips & tricks

- Rule of thumb: the inner query must return **one value** for WHERE/column use, or a
  **temporary result set** (with an alias!) for FROM use.
- The derived-table alias (`AS avg_table`) is mandatory in most engines.

## 🎯 Your turn

1. Employees earning **less than half the max** salary (one statement, no pasted numbers).
2. Show each employee's salary and the max salary as two expression sub-queries.

In [ ]:
# 💡 Solution
q("SELECT f_name, salary FROM employee WHERE salary < (SELECT MAX(salary) FROM employee) / 2")
q("SELECT f_name, salary, (SELECT MAX(salary) FROM employee) AS max_sal, (SELECT MIN(salary) FROM employee) AS min_sal FROM employee LIMIT 4")